# Feature Engineering

## Objective

Create meaningful features from existing employee data to improve data interpretation and potentially enhance the performance of machine learning models.

Unlike data cleaning, feature engineering transforms existing information into more useful representations rather than correcting errors.

In [1]:
import pandas as pd
import numpy as np

# Load Clean Dataset

## Objective

Load the cleaned employee attrition dataset that will be used for feature engineering.

In [2]:
df=pd.read_csv("../data/employee_attrition_clean.csv")

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,3,4,1,6,3,3,2,2,2,2


# Dataset Verification

## Objective

Verify that the dataset has been loaded correctly before creating new features.

In [3]:
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (1470, 32)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeNumber            1470 non-null   int64 
 9   EnvironmentSatisfaction   1470 non-null   int64 
 10  Gender                    1470 non-null   object
 11  HourlyRate                1470 non-null   int64 
 12  JobInvolvement            1470 non-null   int64 
 13  JobLevel                  1470 non-null   int64 
 14

Before creating new features, a copy of the cleaned dataset is created to preserve the original dataset. All feature engineering operations will be performed on this working copy.

In [4]:
df_fe = df.copy()

# Feature 1: Income Group

## Objective

Monthly income is a continuous variable. Grouping employees into salary categories improves interpretability and helps identify salary-related attrition patterns.

In [5]:
df_fe["IncomeGroup"] = pd.qcut(
    df_fe["MonthlyIncome"],
    q=4,
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)

In [6]:
df_fe["IncomeGroup"].value_counts()

IncomeGroup
Low          369
Very High    368
High         367
Medium       366
Name: count, dtype: int64

### Observation

Employees have been grouped into four approximately equal salary categories, making compensation easier to interpret during analysis.

# Feature 2: Experience Group

## Objective

Employees are categorized according to their total work experience to represent different career stages.

In [7]:
df_fe["ExperienceGroup"] = pd.cut(
    df_fe["TotalWorkingYears"],
    bins=[0,5,10,20,40],
    labels=[
        "Early Career",
        "Mid Career",
        "Senior",
        "Highly Experienced"
    ]
)

In [8]:
df_fe["ExperienceGroup"].value_counts()

ExperienceGroup
Mid Career            607
Senior                340
Early Career          305
Highly Experienced    207
Name: count, dtype: int64

### Observation

Employees are grouped into meaningful career stages, allowing experience to be analyzed more effectively than raw numerical values.

# Feature 3: Promotion Delay Category

## Objective

Group employees based on the time since their last promotion to evaluate whether delayed promotions are associated with attrition.

In [9]:
df_fe["PromotionDelay"] = pd.cut(
    df_fe["YearsSinceLastPromotion"],
    bins=[-1,1,3,6,20],
    labels=[
        "Recently Promoted",
        "Moderate Delay",
        "Long Delay",
        "Very Long Delay"
    ]
)

In [10]:
df_fe["PromotionDelay"].value_counts()

PromotionDelay
Recently Promoted    938
Moderate Delay       211
Very Long Delay      183
Long Delay           138
Name: count, dtype: int64

### Observation

Employees are categorized according to promotion waiting time, making promotion-related trends easier to interpret.

# Feature 4: Distance Category

## Objective

Group employees based on commuting distance to simplify the analysis of travel-related attrition patterns.

In [11]:
df_fe["DistanceCategory"] = pd.cut(
    df_fe["DistanceFromHome"],
    bins=[0,5,15,30],
    labels=[
        "Near",
        "Moderate",
        "Far"
    ]
)

In [12]:
df_fe["DistanceCategory"].value_counts()

DistanceCategory
Near        632
Moderate    509
Far         329
Name: count, dtype: int64

### Observation

Employees are grouped according to commuting distance, allowing travel patterns to be analyzed more effectively.

# Feature 5: Overall Satisfaction Score

## Objective

Combine multiple satisfaction-related variables into a single score representing overall employee satisfaction.

In [13]:
df_fe["OverallSatisfaction"] = (
    df_fe["EnvironmentSatisfaction"] +
    df_fe["JobSatisfaction"] +
    df_fe["RelationshipSatisfaction"] +
    df_fe["WorkLifeBalance"]
)

In [14]:
df_fe["OverallSatisfaction"].describe()

count    1470.000000
mean       10.923810
std         2.023259
min         4.000000
25%        10.000000
50%        11.000000
75%        12.000000
max        16.000000
Name: OverallSatisfaction, dtype: float64

### Observation

The combined satisfaction score provides a single measure of employee well-being while preserving information from multiple satisfaction-related variables.

# Feature 6: Company Tenure Group

## Objective

Group employees based on the number of years they have spent at the organization to better analyze tenure-related attrition.

In [15]:
df_fe["TenureGroup"] = pd.cut(
    df_fe["YearsAtCompany"],
    bins=[0,2,5,10,40],
    labels=[
        "New",
        "Junior",
        "Experienced",
        "Veteran"
    ]
)

In [16]:
df_fe["TenureGroup"].value_counts()

TenureGroup
Experienced    448
Junior         434
New            298
Veteran        246
Name: count, dtype: int64

### Observation

Employees are categorized into organizational tenure groups, making comparisons across different stages of employment easier.


# Analyze Engineered Features

## Objective

Examine the distribution of each newly created feature to verify that the feature engineering process has produced meaningful and balanced categories.

In [17]:
df_fe["IncomeGroup"].value_counts()

IncomeGroup
Low          369
Very High    368
High         367
Medium       366
Name: count, dtype: int64

In [18]:
df_fe["ExperienceGroup"].value_counts()

ExperienceGroup
Mid Career            607
Senior                340
Early Career          305
Highly Experienced    207
Name: count, dtype: int64

In [19]:
df_fe["PromotionDelay"].value_counts()

PromotionDelay
Recently Promoted    938
Moderate Delay       211
Very Long Delay      183
Long Delay           138
Name: count, dtype: int64

In [20]:
df_fe["DistanceCategory"].value_counts()

DistanceCategory
Near        632
Moderate    509
Far         329
Name: count, dtype: int64

In [21]:
df_fe["OverallSatisfaction"].describe()

count    1470.000000
mean       10.923810
std         2.023259
min         4.000000
25%        10.000000
50%        11.000000
75%        12.000000
max        16.000000
Name: OverallSatisfaction, dtype: float64

In [22]:
df_fe["TenureGroup"].value_counts()

TenureGroup
Experienced    448
Junior         434
New            298
Veteran        246
Name: count, dtype: int64

### Observation

The engineered features have been successfully created and show meaningful distributions. These features simplify employee segmentation and improve the interpretability of the dataset for subsequent machine learning tasks.


# Verify Engineered Features

## Objective

Verify that all engineered features have been created successfully.

In [23]:
df_fe.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,IncomeGroup,ExperienceGroup,PromotionDelay,DistanceCategory,OverallSatisfaction,TenureGroup
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,6,4,0,5,High,Mid Career,Recently Promoted,Near,8,Experienced
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,10,7,1,7,High,Mid Career,Recently Promoted,Moderate,12,Experienced
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,0,0,0,0,Low,Mid Career,Recently Promoted,Near,12,NaN
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,8,7,3,0,Low,Mid Career,Moderate Delay,Near,13,Experienced
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,2,2,2,2,Medium,Mid Career,Moderate Delay,Near,10,New


In [24]:
df_fe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 38 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   Age                       1470 non-null   int64   
 1   Attrition                 1470 non-null   object  
 2   BusinessTravel            1470 non-null   object  
 3   DailyRate                 1470 non-null   int64   
 4   Department                1470 non-null   object  
 5   DistanceFromHome          1470 non-null   int64   
 6   Education                 1470 non-null   int64   
 7   EducationField            1470 non-null   object  
 8   EmployeeNumber            1470 non-null   int64   
 9   EnvironmentSatisfaction   1470 non-null   int64   
 10  Gender                    1470 non-null   object  
 11  HourlyRate                1470 non-null   int64   
 12  JobInvolvement            1470 non-null   int64   
 13  JobLevel                  1470 non-null   int64 

In [25]:
df_fe.columns

Index(['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department',
       'DistanceFromHome', 'Education', 'EducationField', 'EmployeeNumber',
       'EnvironmentSatisfaction', 'Gender', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus',
       'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'OverTime',
       'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
       'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear',
       'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole',
       'YearsSinceLastPromotion', 'YearsWithCurrManager', 'IncomeGroup',
       'ExperienceGroup', 'PromotionDelay', 'DistanceCategory',
       'OverallSatisfaction', 'TenureGroup'],
      dtype='object')

# Compare Original and Enhanced Dataset

## Objective

Compare the original dataset with the feature-engineered dataset to verify the addition of new variables.

In [26]:
print("Original Shape :", df.shape)

print("Enhanced Shape :", df_fe.shape)

Original Shape : (1470, 32)
Enhanced Shape : (1470, 38)


# Save Feature Engineered Dataset

## Objective

Save the enhanced dataset for use during preprocessing and machine learning model development.

In [27]:
df_fe.to_csv(
    "../data/employee_attrition_feature_engineered.csv",
    index=False
)

# Feature Engineering Summary

The following features were created to improve data interpretation and support predictive modeling.

| Feature | Purpose |
|----------|---------|
| **IncomeGroup** | Categorize employees based on salary level |
| **ExperienceGroup** | Represent different career stages |
| **PromotionDelay** | Measure the waiting period since the last promotion |
| **DistanceCategory** | Categorize commuting distance |
| **OverallSatisfaction** | Combine multiple satisfaction indicators into one score |
| **TenureGroup** | Group employees by years spent at the company |

These engineered features simplify complex numerical variables into meaningful business-oriented categories and provide additional information that may improve both model interpretability and predictive performance.

In [28]:
pd.qcut(
    df_fe["MonthlyIncome"],
    q=4,
    retbins=True
)

(0         (4919.0, 8379.0]
 1         (4919.0, 8379.0]
 2       (1008.999, 2911.0]
 3       (1008.999, 2911.0]
 4         (2911.0, 4919.0]
                ...        
 1465    (1008.999, 2911.0]
 1466     (8379.0, 19999.0]
 1467      (4919.0, 8379.0]
 1468      (4919.0, 8379.0]
 1469      (2911.0, 4919.0]
 Name: MonthlyIncome, Length: 1470, dtype: category
 Categories (4, interval[float64, right]): [(1008.999, 2911.0] < (2911.0, 4919.0] < (4919.0, 8379.0] < (8379.0, 19999.0]],
 array([ 1009.,  2911.,  4919.,  8379., 19999.]))